# 03 — Baseline Model Training & Evaluation (Phase 3)

Compares Logistic Regression vs XGBoost baselines using cross-validation
only. **The test set is not touched in this notebook** — model selection
must happen from CV evidence alone; test-set evaluation is reserved for
`scripts/run_modeling_pipeline.py`'s final step.

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_predict, StratifiedKFold

from src.config import get_default_config
from src.data.load import load_and_validate_data
from src.data.split import train_test_split_data
from src.models.pipeline import build_model_pipeline
from src.models.evaluate import compute_classification_metrics

config = get_default_config()
config.data.target_column = "<set explicitly — dataset-specific>"

In [ ]:
df, schema = load_and_validate_data(config.data)
config.preprocessing.numerical_features = schema.numerical_features
config.preprocessing.categorical_features = schema.categorical_features

X_train, X_test, y_train, y_test = train_test_split_data(
    df, config.data.target_column, config.split
)

## Optional: enable class imbalance handling

Based on the target distribution observed in `01_data_loading_eda.ipynb`.
This is an explicit, human decision — never auto-applied by `src/`.

In [ ]:
# Uncomment only after reviewing 01_data_loading_eda.ipynb's target balance:
# config.model.class_imbalance_strategy = "class_weight"  # or "scale_pos_weight"
print("class_imbalance_strategy:", config.model.class_imbalance_strategy)

## Logistic Regression baseline

In [ ]:
config.model.model_type = "logistic_regression"
lr_pipeline = build_model_pipeline(config.model, config.preprocessing)

cv = StratifiedKFold(
    n_splits=config.cv.n_splits, shuffle=config.cv.shuffle, random_state=config.cv.random_state
)

lr_oof_proba = cross_val_predict(
    lr_pipeline, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

lr_metrics = compute_classification_metrics(y_train, lr_oof_proba)
lr_metrics

## XGBoost baseline

In [ ]:
config.model.model_type = "xgboost"
xgb_pipeline = build_model_pipeline(config.model, config.preprocessing)

xgb_oof_proba = cross_val_predict(
    xgb_pipeline, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

xgb_metrics = compute_classification_metrics(y_train, xgb_oof_proba)
xgb_metrics

## Side-by-side comparison

In [ ]:
comparison = pd.DataFrame({"logistic_regression": lr_metrics, "xgboost": xgb_metrics}).T
comparison

## Edge-case sanity checks on the metrics function

(Mirrors what `tests/test_models_evaluate.py` checks formally — done here
just for visual confirmation.)

In [ ]:
import numpy as np

# Single-class y_true should not crash — metrics that are undefined should
# be reported as such (e.g. NaN), not silently defaulted.
single_class_y = pd.Series([0, 0, 0, 0])
single_class_proba = np.array([0.1, 0.2, 0.3, 0.4])
compute_classification_metrics(single_class_y, single_class_proba)

## Notes / next steps

- Winning baseline (by CV `roc_auc`, then inspect `f1`/`recall` trade-off): `<fill in>`
- Proceed to `04_hyperparameter_tuning_threshold.ipynb` for XGBoost tuning
  (Phase 4) if it's the stronger baseline, per project decision so far.